# ASTRA · Notebook 02 — Replicating PELS-VAE (Martínez-Palomera, Bloom & Abrahams 2020)

**CS787 (Generative AI) project, Group 4.** Runs on Thamilselvan's Kaggle account.

**Goal of this notebook.** Reproduce the secondary base paper as exactly as possible: *Deep Generative Modeling of Periodic Variable Stars Using Physical Parameters*, arXiv 2005.07773. We use **their own data release and their released code's settings**. Everything is re-implemented here, and our port is checked against their code, which gives the same parameter count, 300,897, and the same outputs.

| What we reproduce | Paper |
|---|---|
| Fig. 4: validation loss over training, TCN vs GRU, with/without physical parameters | ≈ 0.011 (TCN), 0.010 (TCN + physics), 0.014 (GRU) at epoch 100 |
| Figs. 5, 7: reconstructions for all 8 classes | shape and class details recovered |
| Fig. 6: the 4-D latent space | close to Gaussian, weak class clustering |
| **Table 3**: regression physical parameters → latent space, validation RMSE | cVAE: linear 0.863, RF **0.299**, MLP 0.863 · cVAE-P: 0.794, **0.289**, 0.798 |
| Fig. 8: RR Lyrae generated at increasing T_eff | sawtooth (RRab) → sinusoidal (RRc) with the RF regressor |

**The method, in the words of the VAE slides.** An **encoder** compresses a light curve (phase, magnitude, error at 600 points) plus its class into the mean and variance of a 4-D Gaussian. A latent vector is sampled with the *reparameterisation trick* (z = μ + σ·ε), and a **decoder** rebuilds the magnitudes from z, the class and the phases. The loss is reconstruction error plus β × the KL divergence to N(0, I), which keeps the latent space smooth enough to sample from. Both the encoder and the decoder are **Temporal Convolutional Networks**: 1-D *causal*, *dilated* convolutions with residual blocks, so each block sees twice as far back as the previous one.

**Paper vs released code (we follow the code, because it produced the paper's figures):**
- The loss is plain MSE + 0.75 · KL / 10⁴. The paper's Eq. 4 instead has an error-weighted MSE and a KL term on the errors; the code comments these out.
- The best model has 9 TCN blocks with kernel 5 (W&B run `b68z1hwo`). The paper's Table 2 marks 7 and 7 in bold.
- Training runs 150 epochs (the paper's text and Fig. 4 say 100).
- The split is a random 80/20 of all 65,760 curves (seed 13). The paper says 60/20/20 with real-only test stars. Because the file mixes augmented copies of the same stars, the code's split leaks copies of validation stars into training. We keep it for exactness and flag it in the analysis.
- The Table 3 MLP gets unscaled inputs (T_eff ≈ 5000), which is why it scores the same as the linear model.

**How to run (Thamilselvan's account)**
- Accelerator **GPU T4 ×2** · Internet **On**.
- First `SMOKE = True` interactively (about 10 min): tests, tiny trainings, and an estimate of each job's hours.
- Then `SMOKE = False` → **Save Version → Save & Run All**. Each GPU works down its queue (measured in session 1: 2.8 h per cVAE, 1.9 h per cVAE-P, 0.5 h for the GRU; 6.2 h in all). Each epoch is checkpointed. `SESSION_HOURS` (10.5) is the time limit of the training, not of the notebook: a job that is not finished pauses about 40 minutes before it, and the run ends when its cells are done.
- If a job is unfinished, or to refresh only the tables and figures: *Add Input* → this notebook's previous output, `SESSION` + 1, run again. Finished models are not trained again; when all five are done no worker starts, and the run only rebuilds the tables and figures (a few minutes). Check the first lines of the log: `restored N file(s)` and the jobs with `'state': 'done'`. If it says `fresh start`, the previous output is not attached: cancel the run, or everything trains again.

In [ ]:
# ---- Configuration ----
import os, sys, json, time, shutil, subprocess
from pathlib import Path

SMOKE = True            # True: tests + tiny jobs + timing (run interactively first); False: the real run
SESSION = 1             # 1, 2, ... only labels this run's summary
SESSION_HOURS = 10.5    # time limit of the training (jobs pause ~40 min before it); it does not end the notebook

# Each GPU trains its queue one model after another (tcn = their best cVAE, tcn_phys = cVAE-P,
# gru = the GRU of Fig. 4, *_s14 = a second seed with the same split)
QUEUES = {
    "0": ["tcn", "gru", "tcn_s14"],
    "1": ["tcn_phys", "tcn_phys_s14"],
}

LOCAL_TEST = os.environ.get("ASTRA_LOCAL_TEST") == "1"     # only my offline test run sets this
RUN_STARTED = time.time()
WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))
RUNS = WORK / ("runs_smoke" if SMOKE else "runs")        # smoke outputs never mix with the real ones
FIGS, TABLES = WORK / "figures", WORK / "tables"
for d in (RUNS, FIGS, TABLES, WORK / "data", WORK / "astra"):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, str(WORK))
print(f"session {SESSION} ({SESSION_HOURS} h), {'SMOKE' if SMOKE else 'REAL RUN'} | queues: {QUEUES}")

In [ ]:
# ---- GPUs, and restoring the previous session's output ----
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
print("torch", torch.__version__, "| GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
assert LOCAL_TEST or torch.cuda.device_count() >= 2, "Set Accelerator to GPU T4 x2"
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def restore_previous():
    """Copies runs/, figures/ and tables/ of an attached previous version (finished jobs are skipped,
    unfinished ones resume from their last epoch). Files already present are kept."""
    n = 0
    for name in ("runs", "figures", "tables"):
        for root in INPUT.glob(f"*/**/{name}"):
            if not root.is_dir() or WORK in root.parents:
                continue
            for src in root.rglob("*"):
                dest = WORK / name / src.relative_to(root)
                if src.is_file() and not dest.exists():
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(src, dest)
                    n += 1
    return n


n = restore_previous()
print(f"restored {n} file(s) from the previous session" if n else "fresh start (nothing to restore)")
for p in sorted(RUNS.glob("*/status.json")):
    print(f"  {p.parent.name:<14} {json.loads(p.read_text())}")

## The `astra` package

The next cells write small modules to `/kaggle/working/astra/`, so the GPU workers (one process per GPU) can import them:
- `common`: paths, seeds, the time guard, safe saving;
- `pelsvae`: their data file, normalisation and split, exactly as in their `datasets.py`;
- `tcn_vae`: the TCN and GRU VAEs and their loss, ported from their `tcn.py`, `vae_models.py` and `vae_training.py`;
- `run_vae`: the per-GPU worker, which trains with checkpointing and saves latent codes and reconstructions.

In [ ]:
%%writefile astra/__init__.py
"""ASTRA: generative models for variable-star light curves (CS787 project, Group 4)."""


In [ ]:
%%writefile astra/common.py
"""Small helpers shared by every notebook: seeds, paths, time guard, safe saving, logging."""
import json
import os
import random
import time
import urllib.request
from pathlib import Path

import numpy as np
import torch

WORK = Path(os.environ.get("ASTRA_WORK", "/kaggle/working"))
INPUT = Path(os.environ.get("ASTRA_INPUT", "/kaggle/input"))


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def save_json(obj, path):
    """Write to a temp file first, so a kill mid-write never leaves a broken file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=1, default=float), encoding="utf-8")
    os.replace(tmp, path)


def load_json(path, default=None):
    path = Path(path)
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else default


def torch_save(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def restore_rng(state, device):
    """Put back the random-number state saved in a resumable checkpoint. torch.load(map_location=GPU) moves
    the saved states to the GPU, but the generators only accept CPU tensors, so they are moved back first.
    Never fatal: without it a resumed run just continues with fresh random numbers. Returns True on success."""
    try:
        torch.set_rng_state(state["rng_cpu"].cpu())
        if state.get("rng_cuda") is not None and str(device).startswith("cuda"):
            torch.cuda.set_rng_state_all([s.cpu() for s in state["rng_cuda"]])
        return True
    except Exception as e:
        print(f"could not restore the random-number state ({type(e).__name__}: {e}); continuing with a fresh one",
              flush=True)
        return False


class Deadline:
    """Session time guard: Kaggle kills a session at 12 h, we stop cleanly before that."""

    def __init__(self, deadline_unix):
        self.t = float(deadline_unix)

    def left(self):
        return self.t - time.time()

    def passed(self, reserve_s=0.0):
        return self.left() < reserve_s


class Logger:
    def __init__(self, path):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)

    def __call__(self, *msg):
        line = time.strftime("%H:%M:%S ") + " ".join(str(m) for m in msg)
        print(line, flush=True)
        with open(self.path, "a", encoding="utf-8") as f:
            f.write(line + "\n")


def find_input(name):
    """A file attached under /kaggle/input (e.g. a previous version's output), if any."""
    hits = sorted(INPUT.glob(f"**/{name}")) if INPUT.exists() else []
    return hits[0] if hits else None


def download(url, dest, tries=5):
    """Download once; reuse a copy attached under /kaggle/input if there is one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    attached = find_input(dest.name)
    if attached is not None:
        return attached
    dest.parent.mkdir(parents=True, exist_ok=True)
    for i in range(tries):
        try:
            tmp = dest.with_suffix(dest.suffix + ".part")
            urllib.request.urlretrieve(url, tmp)
            os.replace(tmp, dest)
            return dest
        except Exception as e:  # network hiccup: wait and retry
            print(f"download failed ({e}), retry {i + 1}/{tries}")
            time.sleep(10 * (i + 1))
    raise RuntimeError(f"could not download {url}")


def peak_gpu_mem_gb():
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.max_memory_allocated() / 1e9


In [ ]:
%%writefile astra/pelsvae.py
"""PELS-VAE data (Martinez-Palomera, Bloom & Abrahams 2020), loaded exactly as their released code does
(github.com/jorgemarpa/PELS-VAE, src/datasets.py and vae_main.py).

The Zenodo training file holds 65,760 OGLE-III I-band light curves of 8 classes: 33,124 real stars
(phase-folded, 300 random points) plus 32,636 augmented copies (magnitudes resampled from N(m, err),
random phase shift, re-sampled). Every curve is shown over 2 phase cycles, so it has 600 points.
Each curve is min-max scaled on its own to [0.0001, 0.9999] (phase and magnitude; errors by the
magnitude range). Their split: a random 80/20 train/validation split with seed 13.
"""
import gzip

import numpy as np
import pandas as pd

from .common import download

ZENODO = "https://zenodo.org/records/3820679/files/"
TRAIN_FILE = "OGLE3_lcs_I_meta_snr5_augmented_folded_trim600.npy.gz"
PHYS = {"P": "Period", "T": "teff_val", "M": "[Fe/H]_J95", "C": "bp_rp", "A": "abs_Gmag", "R": "radius_val",
        "L": "lum_val"}           # their --phy letters, in their order


def load_raw(data_dir):
    path = download(ZENODO + TRAIN_FILE + "?download=1", data_dir / TRAIN_FILE)
    with gzip.open(path, "rb") as f:
        d = np.load(f, allow_pickle=True).item()
    return d["lcs"].astype(np.float32), d["meta"].reset_index(drop=True)


def normalize_each(lcs, lo=0.0001, hi=0.9999):
    """Their utils.normalize_each(norm_time=True, scale_to=[.0001, .9999]), vectorised."""
    out = np.empty_like(lcs)
    t, m, e = lcs[..., 0], lcs[..., 1], lcs[..., 2]
    tr = t.max(1, keepdims=True) - t.min(1, keepdims=True)
    mr = m.max(1, keepdims=True) - m.min(1, keepdims=True)
    out[..., 0] = (t - t.min(1, keepdims=True)) / tr * (hi - lo) + lo
    out[..., 1] = (m - m.min(1, keepdims=True)) / mr * (hi - lo) + lo
    out[..., 2] = e / mr * (hi - lo)
    return out


def make_dataset(lcs_norm, meta, phy=""):
    """Their Astro_lightcurves(...) + remove_nan(): one-hot labels (alphabetical classes), min-max scaled
    physical parameters (scaler fitted on all rows, NaNs ignored), rows with missing parameters dropped."""
    classes = sorted(meta.Type.unique())
    y = meta.Type.map({c: i for i, c in enumerate(classes)}).values.astype(np.int64)
    names = [PHYS[k] for k in "PTMCARL" if k in phy.upper()]
    cols = names if names else ["Period"]
    p = meta[cols].values.astype(np.float32)
    lo, hi = np.nanmin(p, 0), np.nanmax(p, 0)
    p = (p - lo) / (hi - lo)
    keep = ~np.isnan(p).any(1)
    return {"x": lcs_norm[keep], "y": y[keep], "phy": p[keep] if names else np.zeros((keep.sum(), 0), np.float32),
            "rows": np.flatnonzero(keep), "classes": classes, "phy_names": names}


def split_indices(n, seed=13, test_split=0.2):
    """Their get_dataloader(): np.random.seed(seed); shuffle list(range(n)); first 20% = validation."""
    np.random.seed(seed)
    idx = list(range(n))
    np.random.shuffle(idx)
    k = int(np.floor(test_split * n))
    return np.array(idx[k:]), np.array(idx[:k])


def class_counts(meta):
    real = meta.drop_duplicates("OGLE_id")
    return pd.DataFrame({"curves (with augmented)": meta.Type.value_counts().sort_index(),
                         "unique stars": real.Type.value_counts().sort_index()})


In [ ]:
%%writefile astra/tcn_vae.py
"""PELS-VAE models, ported line by line from their src/tcn.py and src/vae_models.py.

Encoder: a Temporal Convolutional Network (TCN: causal, dilated convolutions with residual blocks) reads
(phase, magnitude, error) of the light curve; its last time step, concatenated with the one-hot class
(and, for cVAE-P, the physical parameters), goes through Dense(16)+ReLU+Dropout to the mean and
log-variance of a small latent space (4 dimensions). Decoder: the latent vector and class are repeated at
every time step next to the phase, a second TCN runs over them, and Dense+sigmoid gives the magnitude.
A GRU version (VAE_RNN) replaces both TCNs by recurrent layers (their Fig. 4 comparison).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.parametrizations import weight_norm


class Chomp1d(nn.Module):
    """Drops the extra right-hand padding, which makes the convolution causal."""

    def __init__(self, n):
        super().__init__()
        self.n = n

    def forward(self, x):
        return x[:, :, :-self.n].contiguous()


class TemporalBlock(nn.Module):
    def __init__(self, n_in, n_out, k, dilation, dropout):
        super().__init__()
        pad = (k - 1) * dilation
        self.net = nn.Sequential(
            weight_norm(nn.Conv1d(n_in, n_out, k, padding=pad, dilation=dilation)), Chomp1d(pad), nn.ReLU(),
            nn.Dropout(dropout),
            weight_norm(nn.Conv1d(n_out, n_out, k, padding=pad, dilation=dilation)), Chomp1d(pad), nn.ReLU(),
            nn.Dropout(dropout))
        self.downsample = nn.Conv1d(n_in, n_out, 1) if n_in != n_out else None
        # their init_weights() sets conv weights ~ N(0, 0.01), but on weight-normed convs that assignment is
        # overwritten by the next forward pass, so only the 1x1 residual conv really gets it
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        res = x if self.downsample is None else self.downsample(x)
        return F.relu(self.net(x) + res)


class TemporalConvNet(nn.Module):
    def __init__(self, n_in, channels, k, dropout):
        super().__init__()
        self.network = nn.Sequential(*[TemporalBlock(n_in if i == 0 else channels[i - 1], channels[i], k, 2 ** i,
                                                     dropout) for i in range(len(channels))])

    def forward(self, x):
        return self.network(x)


class VAE_TCN(nn.Module):
    def __init__(self, latent_dim=4, seq_len=600, kernel_size=5, hidden_dim=48, nlevels=9, n_feats=3,
                 dropout=0.2, lab_dim=8, phy_dim=0):
        super().__init__()
        self.latent_dim, self.lab_dim, self.phy_dim, self.seq_len = latent_dim, lab_dim, phy_dim, seq_len
        enc = [hidden_dim] * nlevels
        enc[0], enc[1], enc[2] = 8, 16, 32
        dec = [hidden_dim] * nlevels
        dec[-1], dec[-2], dec[-3] = 8, 16, 32
        self.enc_tcn = TemporalConvNet(n_feats, enc, kernel_size, dropout)
        self.enc_linear = nn.Sequential(nn.Linear(hidden_dim + lab_dim + phy_dim, 16), nn.ReLU(), nn.Dropout(dropout))
        self.enc_linear_mu = nn.Linear(16, latent_dim)
        self.enc_linear_std = nn.Linear(16, latent_dim)           # (it outputs the log-variance)
        self.dec_tcn = TemporalConvNet(1 + latent_dim + lab_dim, dec, kernel_size, dropout)   # feed_pp = False
        self.dec_linear = nn.Linear(dec[-1], 1)

    def encoder(self, x, label, phy=None):
        h = self.enc_tcn(x.transpose(1, 2)).transpose(1, 2)[:, -1, :]
        h = torch.cat([h, label] + ([phy] if self.phy_dim > 0 else []), 1)
        h = self.enc_linear(h)
        return self.enc_linear_mu(h), self.enc_linear_std(h)

    def decoder(self, z, dt, label):
        z = torch.cat([z, label], 1).unsqueeze(1).repeat(1, dt.shape[1], 1)
        h = torch.cat([dt.unsqueeze(-1), z], -1)
        h = self.dec_tcn(h.transpose(1, 2)).transpose(1, 2)
        return torch.sigmoid(self.dec_linear(h))

    def forward(self, x, label, phy=None):
        mu, logvar = self.encoder(x, label, phy)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)       # reparameterisation trick
        return self.decoder(z, x[:, :, 0], label), mu, logvar, z


class VAE_RNN(nn.Module):
    def __init__(self, latent_dim=4, seq_len=600, hidden_dim=48, n_layers=2, n_feats=3, dropout=0.2, lab_dim=8,
                 phy_dim=0):
        super().__init__()
        self.latent_dim, self.lab_dim, self.phy_dim, self.seq_len = latent_dim, lab_dim, phy_dim, seq_len
        self.enc_lstm = nn.GRU(n_feats, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        self.dec_lstm = nn.GRU(1 + latent_dim + lab_dim, hidden_dim, n_layers, batch_first=True, dropout=dropout)
        self.enc_linear = nn.Sequential(nn.Linear(hidden_dim + lab_dim + phy_dim, 16), nn.ReLU(), nn.Dropout(dropout))
        self.enc_linear_mu = nn.Linear(16, latent_dim)
        self.enc_linear_std = nn.Linear(16, latent_dim)
        self.dec_linear = nn.Linear(hidden_dim, 1)
        for rnn in (self.enc_lstm, self.dec_lstm):            # their init_weights()
            for name, p in rnn.named_parameters():
                if "bias" in name:
                    nn.init.normal_(p, 0.0)                  # (std 1, as in their call)
                elif "weight" in name:
                    nn.init.xavier_normal_(p)

    def encoder(self, x, label, phy=None):
        h, _ = self.enc_lstm(x)
        h = torch.cat([h[:, -1, :], label] + ([phy] if self.phy_dim > 0 else []), 1)
        h = self.enc_linear(h)
        return self.enc_linear_mu(h), self.enc_linear_std(h)

    def decoder(self, z, dt, label):
        z = torch.cat([z, label], 1).unsqueeze(1).repeat(1, dt.shape[1], 1)
        h, _ = self.dec_lstm(torch.cat([dt.unsqueeze(-1), z], -1))
        return torch.sigmoid(self.dec_linear(h))

    def forward(self, x, label, phy=None):
        mu, logvar = self.encoder(x, label, phy)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
        return self.decoder(z, x[:, :, 0], label), mu, logvar, z


def vae_loss(x, xhat, mu, logvar, beta):
    """Their Trainer._loss: plain MSE on the scaled magnitude + beta * KL / batch / 1e4.
    (The paper's Eq. 4 has an error-weighted MSE and a third KL term on the errors; the released code
    comments both out, with the note that plain MSE converges faster and better.)"""
    mse = F.mse_loss(xhat[:, :, 0], x[:, :, 1])
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.shape[0] / 1e4
    return mse + beta * kld, mse, kld


In [ ]:
%%writefile astra/run_vae.py
"""Worker for the VAE notebook: trains the PELS-VAE models of a queue on one GPU, skipping finished ones.

python -m astra.run_vae --queue tcn,gru --deadline <unix time> [--smoke] [--runs-dir DIR]
"""
import argparse
import math
import os
import time
import traceback
from pathlib import Path

import numpy as np
import torch

from .common import WORK, Deadline, Logger, load_json, restore_rng, save_json, torch_save
from .pelsvae import load_raw, make_dataset, normalize_each, split_indices
from .tcn_vae import VAE_RNN, VAE_TCN, vae_loss

# The hyper-parameters of their best run (W&B run b68z1hwo, printed in their notebook). The paper's Table 2
# marks 7 blocks and kernel 7 in bold, but the model behind their figures used 9 blocks and kernel 5.
HP = dict(latent=4, hidden=48, levels=9, kernel=5, dropout=0.2, beta=0.75, lr=1e-3, batch=128, epochs=150,
          t_max=50, lr_min=1e-5, rnn_layers=2, split_seed=13)
JOBS = {
    "tcn": dict(arch="tcn", phy="", seed=13),              # cVAE: class-conditional (their best model)
    "tcn_phys": dict(arch="tcn", phy="PTA", seed=13),      # cVAE-P: + period, Teff, absolute G mag in the encoder
    "gru": dict(arch="gru", phy="", seed=13),              # the GRU curve of their Fig. 4
    "tcn_s14": dict(arch="tcn", phy="", seed=14),          # second seeds (same split, new init and batch order)
    "tcn_phys_s14": dict(arch="tcn", phy="PTA", seed=14),
}
SMOKE = dict(epochs=2, n_rows=3000)          # tiny jobs: a random 3,000-curve subset, 2 epochs


def cosine_lr(epoch, c):
    """Their CosineAnnealingLR(T_max=50, eta_min=1e-5) stepped as sch.step(epoch): the closed form, so over
    150 epochs the rate goes down, up and down again."""
    return c["lr_min"] + (c["lr"] - c["lr_min"]) * (1 + math.cos(math.pi * epoch / c["t_max"])) / 2


def build_model(c, n_classes, phy_dim):
    if c["arch"] == "tcn":
        return VAE_TCN(c["latent"], 600, c["kernel"], c["hidden"], c["levels"], 3, c["dropout"], n_classes, phy_dim)
    return VAE_RNN(c["latent"], 600, c["hidden"], c["rnn_layers"], 3, c["dropout"], n_classes, phy_dim)


def run_job(c, X, Y, P, tr, va, out, deadline, dev, log, n_classes=8):
    torch.manual_seed(c["seed"])
    onehot = torch.eye(n_classes, device=dev)
    model = build_model(c, n_classes, P.shape[1]).to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=c["lr"])
    hist, start = [], 1
    if (out / "last.pt").exists():
        s = torch.load(out / "last.pt", map_location=dev, weights_only=False)
        model.load_state_dict(s["model"]); opt.load_state_dict(s["opt"])
        hist, start = s["hist"], s["epoch"] + 1
        restore_rng(s, dev)
        log(f"resumed after epoch {s['epoch']}")
    log(f"{c['arch']} phy='{c['phy']}': {sum(p.numel() for p in model.parameters()):,} parameters, "
        f"train {len(tr):,} / val {len(va):,}")

    def batches(idx, shuffle, epoch):
        if shuffle:
            g = torch.Generator().manual_seed(c["seed"] * 1000 + epoch)
            idx = idx[torch.randperm(len(idx), generator=g).numpy()]
        for i in range(0, len(idx), c["batch"]):
            j = torch.from_numpy(idx[i:i + c["batch"]]).to(dev)
            yield X[j], onehot[Y[j]], P[j]

    for epoch in range(start, c["epochs"] + 1):
        if deadline.passed(900):
            log(f"time budget reached before epoch {epoch}: resume next session")
            return "paused"
        t0 = time.time()
        for g in opt.param_groups:
            g["lr"] = c["lr"] if epoch == 1 else cosine_lr(epoch - 1, c)
        model.train()
        sums = torch.zeros(3, device=dev); n = 0
        for x, lab, p in batches(tr, True, epoch):
            xhat, mu, logvar, _ = model(x, lab, p)
            loss, mse, kld = vae_loss(x, xhat, mu, logvar, c["beta"])
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sums += torch.stack([loss.detach(), mse.detach(), kld.detach()]); n += 1
        tr_l = (sums / n).tolist()
        model.eval()
        sums = torch.zeros(3, device=dev); n = 0
        with torch.no_grad():                       # like their _test_epoch: eval mode, z still sampled
            for x, lab, p in batches(va, False, epoch):
                xhat, mu, logvar, _ = model(x, lab, p)
                sums += torch.stack(vae_loss(x, xhat, mu, logvar, c["beta"])); n += 1
        va_l = (sums / n).tolist()
        hist.append({"epoch": epoch, "lr": opt.param_groups[0]["lr"], "train_loss": tr_l[0], "train_mse": tr_l[1],
                     "train_kl": tr_l[2], "val_loss": va_l[0], "val_mse": va_l[1], "val_kl": va_l[2],
                     "seconds": time.time() - t0})
        torch_save({"model": model.state_dict(), "opt": opt.state_dict(), "hist": hist, "epoch": epoch,
                    "rng_cpu": torch.get_rng_state(),
                    "rng_cuda": torch.cuda.get_rng_state_all() if dev == "cuda" else None}, out / "last.pt")
        save_json(hist, out / "history.json")
        if epoch == 1 or epoch % 10 == 0 or epoch == c["epochs"]:
            log(f"epoch {epoch}: train loss {tr_l[0]:.5f} (mse {tr_l[1]:.5f}, kl {tr_l[2]:.5f}) | "
                f"val loss {va_l[0]:.5f} | lr {hist[-1]['lr']:.2e} | {hist[-1]['seconds']:.0f} s")
    torch_save(model.state_dict(), out / "model.pt")
    # latent means and std of every curve (dataset order), used for Fig. 6 and the regressions of Table 3
    mus, stds = [], []
    with torch.no_grad():
        allidx = np.arange(len(Y))
        for x, lab, p in batches(allidx, False, 0):
            mu, logvar = model.encoder(x, lab, p)
            mus.append(mu.cpu()); stds.append(torch.exp(0.5 * logvar).cpu())
        # reconstructions from the latent mean for 3 validation curves per class (their Figs. 5 and 7)
        rng = np.random.default_rng(0)
        yva = Y[torch.from_numpy(va).to(dev)].cpu().numpy()
        show = np.concatenate([rng.choice(va[yva == k], min(3, int(np.sum(yva == k))), replace=False)
                               for k in range(n_classes)])
        j = torch.from_numpy(show).to(dev)
        mu, _ = model.encoder(X[j], onehot[Y[j]], P[j])
        rec = model.decoder(mu, X[j][:, :, 0], onehot[Y[j]])
    np.savez(out / "latent.npz", mu=torch.cat(mus).numpy(), std=torch.cat(stds).numpy())
    np.savez(out / "recon.npz", idx=show, x=X[j].cpu().numpy(), xhat=rec[:, :, 0].cpu().numpy())
    (out / "last.pt").unlink(missing_ok=True)        # keep only the final weights
    return "done"


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--queue", required=True)
    ap.add_argument("--deadline", type=float, required=True)
    ap.add_argument("--smoke", action="store_true")
    ap.add_argument("--runs-dir", default=str(WORK / "runs"))
    a = ap.parse_args()
    runs_dir = Path(a.runs_dir)
    tiny = a.smoke or os.environ.get("ASTRA_LOCAL_TEST") == "1"      # my offline CPU test runs tiny jobs
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    deadline = Deadline(a.deadline)
    torch.backends.cudnn.benchmark = True
    lcs, meta = load_raw(WORK / "data")
    lcs = normalize_each(lcs)
    for job in a.queue.split(","):
        status_path = runs_dir / job / "status.json"
        if (load_json(status_path, {}) or {}).get("state") == "done":
            print(f"[{job}] already done, skipping", flush=True)
            continue
        out = runs_dir / job
        log = Logger(out / "log.txt")
        c = dict(HP, **JOBS[job], **(SMOKE if tiny else {}))
        save_json(c, out / "config.json")
        log(f"start {job} ({'tiny' if tiny else 'real run'})")
        t0 = time.time()
        if dev == "cuda":
            torch.cuda.reset_peak_memory_stats()
        try:
            ds = make_dataset(lcs, meta, c["phy"])
            if tiny:
                sub = np.sort(np.random.default_rng(0).choice(len(ds["y"]), c["n_rows"], replace=False))
                ds = dict(ds, **{k: ds[k][sub] for k in ("x", "y", "phy", "rows")})
            tr, va = split_indices(len(ds["y"]), c["split_seed"])
            np.savez(out / "split.npz", rows=ds["rows"], train=tr, val=va)
            X = torch.from_numpy(ds["x"]).to(dev)
            Y = torch.from_numpy(ds["y"]).to(dev)
            P = torch.from_numpy(ds["phy"]).to(dev)
            state = run_job(c, X, Y, P, tr, va, out, deadline, dev, log, len(ds["classes"]))
            del X, Y, P
        except Exception:
            log(traceback.format_exc())
            state = "failed"
        prev = load_json(status_path, {}) or {}
        save_json({"state": state, "hours_this_session": (time.time() - t0) / 3600,
                   "hours_total": prev.get("hours_total", 0) + (time.time() - t0) / 3600,
                   "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if dev == "cuda" else 0}, status_path)
        log(f"{job}: {state}")
        if state == "paused":
            break


if __name__ == "__main__":
    main()


## 1 · Data: their Zenodo release (OGLE-III I-band + Gaia DR2)

`OGLE3_lcs_I_meta_snr5_augmented_folded_trim600.npy.gz` (296 MB) is the exact file their model was trained on:
- **Light curves:** OGLE-III, with S/N ≥ 5, 3 rounds of 3σ clipping and multi-mode sub-types removed. Each is cross-matched to Gaia DR2 and validated, then phase-folded and randomly sub-sampled to 300 points.
- **Augmentation:** rare classes are topped up to 5,000 and common ones to about 10,000 curves (their Table 1, column 5). Each copy re-draws every magnitude from N(m, error), shifts the phase at random and re-samples 300 points.
- **Two cycles:** every curve is repeated over phase 0 to 2, so it has 600 points.
- **Physical parameters:** the metadata carries Gaia's T_eff, absolute G magnitude, colour, radius and luminosity, next to OGLE's period.

The table below checks the file against their Table 1 (clean-validated stars and augmented totals).

In [ ]:
# ---- Download (about 1 minute) and check against their Table 1 ----
import socket
from astra import common, pelsvae

have = (WORK / "data" / pelsvae.TRAIN_FILE).exists() or common.find_input(pelsvae.TRAIN_FILE) is not None
print("training file:", "found among the inputs" if have else "not attached, downloading 0.3 GB from Zenodo", flush=True)
socket.setdefaulttimeout(120)            # a stalled download raises and is retried; it cannot hang the session
lcs_raw, meta = pelsvae.load_raw(WORK / "data")
socket.setdefaulttimeout(None)
counts = pelsvae.class_counts(meta)
counts["paper: validated"] = [71, 7121, 42, 9505, 2269, 4349, 9322, 436]        # Table 1, column 4
counts["paper: augmented"] = [5000, 10045, 5090, 10000, 10365, 10044, 10169, 5047]   # Table 1, column 5
counts.to_csv(TABLES / "table1_class_counts.csv")
print(counts.to_string())
print(f"\n{len(meta):,} curves x {lcs_raw.shape[1]} points x (phase, mag, err) | "
      f"rows with period, T_eff and abs. G mag: {meta.dropna(subset=['Period', 'teff_val', 'abs_Gmag']).shape[0]:,} "
      f"({meta.dropna(subset=['Period', 'teff_val', 'abs_Gmag']).OGLE_id.nunique():,} unique stars)")

lcs = pelsvae.normalize_each(lcs_raw)
assert lcs[..., :2].min() >= 1e-4 - 1e-7 and lcs[..., :2].max() <= 0.9999 + 1e-6
classes = sorted(meta.Type.unique())
fig, ax = plt.subplots(2, 8, figsize=(22, 4.5), sharex=True)
rng = np.random.default_rng(0)
for k, c in enumerate(classes):
    i = rng.choice(np.flatnonzero(meta.Type.values == c))
    ax[0, k].plot(lcs_raw[i, :, 0], lcs_raw[i, :, 1], ".", ms=2); ax[0, k].invert_yaxis()
    ax[0, k].set_title(f"{c} (P = {meta.Period.values[i]:.3g} d)", fontsize=9)
    ax[1, k].plot(lcs[i, :, 0], lcs[i, :, 1], ".", ms=2, c="C1"); ax[1, k].invert_yaxis()
ax[0, 0].set_ylabel("I mag"); ax[1, 0].set_ylabel("scaled mag (model input)")
fig.supxlabel("phase (two cycles)"); fig.tight_layout(); fig.savefig(FIGS / "data_examples.png", dpi=110); plt.show()
del lcs_raw

## 2 · Self-tests

- Our TCN-VAE with their best settings has **300,897 parameters**, the number their W&B run reports. The GRU and the physics-conditioned versions build too.
- The split reproduces their procedure: 80/20 of all rows with seed 13, and 20% of the 43,031 rows that have all three physical parameters for cVAE-P.
- The encoder, the reparameterisation and the decoder give the right shapes, and the loss is positive.
- Their cosine schedule, stepped with the epoch number, goes down, up and down again over 150 epochs (plotted).

In [ ]:
# ---- Self-tests ----
from astra import tcn_vae, run_vae

count = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)
tcn = tcn_vae.VAE_TCN(4, 600, 5, 48, 9, 3, 0.2, 8, 0)
assert count(tcn) == 300_897, count(tcn)
tcn_p = tcn_vae.VAE_TCN(4, 600, 5, 48, 9, 3, 0.2, 8, 3)
gru = tcn_vae.VAE_RNN(4, 600, 48, 2, 3, 0.2, 8, 0)
print(f"parameters: TCN {count(tcn):,} (their run: 300,897) | TCN + physics {count(tcn_p):,} | GRU {count(gru):,}")

ds = pelsvae.make_dataset(lcs, meta, "")
tr, va = pelsvae.split_indices(len(ds["y"]))
dsp = pelsvae.make_dataset(lcs, meta, "PTA")
trp, vap = pelsvae.split_indices(len(dsp["y"]))
assert len(ds["y"]) == 65_760 and len(dsp["y"]) == 43_031 and len(set(tr) & set(va)) == 0
print(f"cVAE: train {len(tr):,} / val {len(va):,} | cVAE-P ({dsp['phy_names']}): train {len(trp):,} / val {len(vap):,}")
leak = len(set(meta.OGLE_id.values[va]) & set(meta.OGLE_id.values[tr])) / len(set(meta.OGLE_id.values[va]))
print(f"validation stars that also have a copy in training (their split): {100 * leak:.1f}%")

x = torch.from_numpy(ds["x"][:4]); lab = torch.eye(8)[torch.from_numpy(ds["y"][:4])]
for m, p in ((tcn, None), (gru, None), (tcn_p, torch.from_numpy(dsp["phy"][:4]))):
    xhat, mu, logvar, z = m(x, lab, p)
    assert xhat.shape == (4, 600, 1) and mu.shape == (4, 4)
    loss, mse, kld = tcn_vae.vae_loss(x, xhat, mu, logvar, 0.75)
    assert loss.item() > 0
lrs = [run_vae.HP["lr"]] + [run_vae.cosine_lr(e - 1, run_vae.HP) for e in range(2, 151)]
plt.figure(figsize=(5, 2.5)); plt.plot(range(1, 151), lrs); plt.xlabel("epoch"); plt.ylabel("learning rate")
plt.title("their cosine schedule, stepped with the epoch"); plt.tight_layout(); plt.show()
print("all self-tests passed")
del tcn, tcn_p, gru, ds, dsp

In [ ]:
# ---- SMOKE only: time one training epoch of each architecture and estimate the queues ----
if SMOKE and DEV == "cuda":
    X = torch.from_numpy(lcs[:128 * 40]).to(DEV)
    lab = torch.eye(8, device=DEV)[torch.zeros(128, dtype=torch.long, device=DEV)]
    per_epoch = {}
    for arch, m in (("tcn", tcn_vae.VAE_TCN(4, 600, 5, 48, 9, 3, 0.2, 8, 0)),
                    ("gru", tcn_vae.VAE_RNN(4, 600, 48, 2, 3, 0.2, 8, 0))):
        m = m.to(DEV); opt = torch.optim.Adam(m.parameters(), 1e-3)
        for i in range(45):
            if i == 5:
                torch.cuda.synchronize(); t = time.time()
            xb = X[(i % 40) * 128:(i % 40 + 1) * 128]
            loss = tcn_vae.vae_loss(xb, *m(xb, lab)[:3], 0.75)[0]
            opt.zero_grad(); loss.backward(); opt.step()
        torch.cuda.synchronize()
        step = (time.time() - t) / 40
        per_epoch[arch] = step * (52_608 / 128) * 1.15              # + validation
        print(f"{arch}: {step * 1000:.0f} ms/step -> ≈ {per_epoch[arch] / 60:.1f} min/epoch -> "
              f"≈ {150 * per_epoch[arch] / 3600:.1f} h for 150 epochs")
    for gpu, q in QUEUES.items():
        h = sum(150 * per_epoch["gru" if j.startswith("gru") else "tcn"] / 3600 for j in q)
        print(f"GPU {gpu}: {q} ≈ {h:.1f} h")
    json.dump(per_epoch, open(TABLES / "benchmark.json", "w"))
    del X, m, opt

## 3 · Training: one worker per GPU

Each GPU runs `python -m astra.run_vae` on its queue, with the settings of their best run: Adam at lr 10⁻³ with their cosine schedule, batch 128, 150 epochs, β = 0.75, latent size 4, dropout 0.2. After every epoch the worker records the training and validation loss (and its MSE and KL parts) and saves a resumable checkpoint.

After the last epoch, it saves the final weights, the latent mean and standard deviation of all 65,760 curves (for Fig. 6 and Table 3), and reconstructions of 3 validation curves per class (Figs. 5 and 7). In the real run, a short canary with tiny jobs runs first.

Time: a worker pauses its job 15 minutes before the limit (`SESSION_HOURS` minus 0.4 h, kept for the results cells) and the next session continues it. A worker that is still alive at the limit is stuck and is stopped, so the results cells always run. When every job is already done, no worker starts.

In [ ]:
# ---- Launch the two GPU workers and wait ----
del lcs                                  # the workers load their own copy
RESULTS_RESERVE_H = 0.4                  # kept free for the regressions, figures and output saving
deadline = RUN_STARTED + SESSION_HOURS * 3600 - RESULTS_RESERVE_H * 3600


def launch(queues, runs_dir, smoke):
    procs = {}
    for gpu, q in queues.items():
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, PYTHONPATH=str(WORK))
        f = open(runs_dir / f"worker_gpu{gpu}.log", "a")
        cmd = [sys.executable, "-m", "astra.run_vae", "--queue", ",".join(q), "--deadline", str(deadline),
               "--runs-dir", str(runs_dir)] + (["--smoke"] if smoke else [])
        procs[gpu] = (subprocess.Popen(cmd, env=env, stdout=f, stderr=subprocess.STDOUT, cwd=WORK), q)
    return procs


def last_line(job, runs_dir):
    p = runs_dir / job / "log.txt"
    lines = p.read_text(encoding="utf-8").strip().splitlines() if p.exists() else []
    return lines[-1][:150] if lines else "(not started)"


def job_state(job, runs_dir):
    p = runs_dir / job / "status.json"
    return json.loads(p.read_text())["state"] if p.exists() else ""


def wait(procs, runs_dir, every_s, stop_at):
    """Returns as soon as every worker has exited and prints a status line every `every_s` seconds.
    A worker pauses its job by itself 15 minutes before the deadline, so one that is still alive at `stop_at`
    is stuck: it is stopped here (its last saved epoch is kept) and the session goes on to the results cells
    instead of running into Kaggle's 12-hour limit, where the whole output would be lost."""
    t_print = time.time()
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(5)
        if time.time() > stop_at:
            for gpu, (p, _) in procs.items():
                if p.poll() is None:
                    p.kill()
                    p.wait()
                    print(f"!! the worker on GPU {gpu} was still running at the time limit and was stopped "
                          f"(see {runs_dir.name}/worker_gpu{gpu}.log); its last saved state is kept", flush=True)
        over = all(p.poll() is not None for p, _ in procs.values())
        if over or time.time() - t_print >= every_s:
            t_print = time.time()
            h = (time.time() - RUN_STARTED) / 3600
            for gpu, (p, q) in procs.items():
                cur = next((j for j in q if job_state(j, runs_dir) not in ("done", "failed")), q[-1])
                print(f"[{h:5.2f} h] GPU {gpu} {'running' if p.poll() is None else 'exited '} | {cur}: "
                      f"{last_line(cur, runs_dir)}", flush=True)
    return {gpu: p.returncode for gpu, (p, _) in procs.items()}


todo = [j for q in QUEUES.values() for j in q if job_state(j, RUNS) != "done"]
if not todo:           # a results-only session: no canary, no workers
    print("every job is already done: nothing to train; the results cells use the restored files")
else:
    if not SMOKE:      # canary: tiny versions of each GPU's first job, in a throw-away folder
        canary = WORK / "canary"
        canary.mkdir(exist_ok=True)
        codes = wait(launch({g: q[:1] for g, q in QUEUES.items()}, canary, True), canary, 20, time.time() + 900)
        states = {j.name: json.loads((j / "status.json").read_text())["state"] for j in canary.iterdir()
                  if (j / "status.json").exists()}
        assert all(c == 0 for c in codes.values()) and all(s == "done" for s in states.values()), \
            f"canary failed: {codes} {states} (see {canary})"
        shutil.rmtree(canary)
        print("canary passed\n")
    print(f"jobs to run: {todo} | the workers pause their jobs by {(deadline - 900 - RUN_STARTED) / 3600:.2f} h "
          f"and are stopped at {(deadline - RUN_STARTED) / 3600:.2f} h at the latest")
    codes = wait(launch(QUEUES, RUNS, SMOKE), RUNS, 30 if SMOKE else 600, deadline)
    print("workers finished with exit codes", codes)
for p in sorted(RUNS.glob("*/status.json")):
    print(f"  {p.parent.name:<14} {json.loads(p.read_text())}")

## 4 · Results: Figs. 4–7 (ours vs paper)

- **Fig. 4**: validation loss per epoch, with the paper's approximate epoch-100 values marked. Training and validation loss are shown side by side for every model, to check for over- or under-fitting. Remember that the validation curves share stars with the training set (their split).
- The table also gives each model's lowest validation loss and its epoch: their schedule raises the learning rate again after epoch 50, and a model can get worse after that.
- **Figs. 5 and 7**: reconstructions from the latent *mean*, for 3 validation curves per class, with and without physical parameters, and for the second seed.
- **Fig. 6**: the latent means of all curves, coloured by class. We also report how close the latent codes are to the N(0, I) prior. With the KL term divided by 10⁴ we expect small posterior σ, which matters later: sampling new curves from the prior is only safe if the codes actually fill N(0, I).

In [ ]:
# ---- Figs. 4-7 from the finished jobs ----
load = lambda p: json.loads(p.read_text()) if p.exists() else None
jobs = [j for q in QUEUES.values() for j in q]
H = {j: load(RUNS / j / "history.json") for j in jobs}
H = {j: h for j, h in H.items() if h}
PAPER_FIG4 = {"tcn": 0.011, "tcn_phys": 0.010, "gru": 0.014}          # read off their Fig. 4 at epoch 100
rows = []
for j, h in H.items():
    at = lambda e, k: next((x[k] for x in h if x["epoch"] == e), np.nan)
    rows.append({"job": j, "epochs": len(h), "val loss @100": at(100, "val_loss"), "val loss @150": at(150, "val_loss"),
                 "paper @100 (Fig. 4)": PAPER_FIG4.get(j, np.nan),
                 "min val loss": min(x["val_loss"] for x in h), "at epoch": min(h, key=lambda x: x["val_loss"])["epoch"],
                 "train loss (last)": h[-1]["train_loss"],
                 "val MSE (last)": h[-1]["val_mse"], "val KL/1e4 (last)": h[-1]["val_kl"],
                 "min/epoch": np.mean([x["seconds"] for x in h]) / 60})
fig4 = pd.DataFrame(rows).round(5)
fig4.to_csv(TABLES / "fig4_losses.csv", index=False)
print(fig4.to_string(index=False) if len(fig4) else "(no finished jobs yet)")

if H:
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for j, h in H.items():
        e = [x["epoch"] for x in h]
        ls = "--" if "s14" in j else "-"
        ax[0].plot(e, [x["val_loss"] for x in h], ls, label=j)
        ax[1].plot(e, [x["train_loss"] for x in h], ls, label=f"{j} train")
        ax[1].plot(e, [x["val_loss"] for x in h], ":", label=f"{j} val")
    for j, v in PAPER_FIG4.items():
        ax[0].scatter([100], [v], marker="x", s=80, label=f"paper {j} @100")
    ax[0].set_ylim(0, 0.06); ax[0].set_title("validation loss (their Fig. 4)"); ax[0].legend(fontsize=7)
    ax[1].set_yscale("log"); ax[1].set_title("train vs validation (over/under-fitting)"); ax[1].legend(fontsize=6)
    for a in ax:
        a.set_xlabel("epoch")
    fig.tight_layout(); fig.savefig(FIGS / "fig4_val_loss.png", dpi=110); plt.show()

# ---- reconstructions (Figs. 5 and 7) ----
classes = sorted(meta.Type.unique())
for j in ("tcn", "tcn_s14", "tcn_phys"):
    p = RUNS / j / "recon.npz"
    if not p.exists():
        continue
    R = np.load(p)
    split = np.load(RUNS / j / "split.npz")
    y = meta.Type.values[split["rows"][R["idx"]]]
    fig, ax = plt.subplots(len(classes), 3, figsize=(12, 2 * len(classes)), sharex=True)
    for k, c in enumerate(classes):
        for r, i in enumerate(np.flatnonzero(y == c)[:3]):
            ax[k, r].plot(R["x"][i, :, 0], R["x"][i, :, 1], ".", c="0.6", ms=2)
            ax[k, r].plot(R["x"][i, :, 0], R["xhat"][i], c="C0", lw=1.2)
            ax[k, r].invert_yaxis(); ax[k, r].set_ylabel(c, fontsize=8)
    fig.suptitle(f"{j}: observed (grey) and reconstructed from the latent mean (blue)")
    fig.tight_layout(); fig.savefig(FIGS / f"recon_{j}.png", dpi=100); plt.show()

# ---- latent space (Fig. 6) and how close it is to the N(0, I) prior ----
if (RUNS / "tcn" / "latent.npz").exists():
    Lz = np.load(RUNS / "tcn" / "latent.npz")
    rows_t = np.load(RUNS / "tcn" / "split.npz")["rows"]
    yz = meta.Type.values[rows_t]
    sub = np.random.default_rng(0).choice(len(yz), min(6000, len(yz)), replace=False)
    d = Lz["mu"].shape[1]
    fig, ax = plt.subplots(d, d, figsize=(11, 11))
    for a in range(d):
        for b in range(d):
            if a == b:
                for c in classes:
                    ax[a, b].hist(Lz["mu"][sub][yz[sub] == c, a], bins=40, histtype="step", label=c)
            elif a > b:
                for c in classes:
                    m = yz[sub] == c
                    ax[a, b].scatter(Lz["mu"][sub][m, b], Lz["mu"][sub][m, a], s=2, alpha=.5)
            else:
                ax[a, b].axis("off")
    ax[0, 0].legend(fontsize=7, loc="upper left", bbox_to_anchor=(1.1, 1))
    fig.suptitle("tcn: latent means mu (their Fig. 6)"); fig.tight_layout()
    fig.savefig(FIGS / "fig6_latent.png", dpi=100); plt.show()
    stats = pd.DataFrame({"mean of mu": Lz["mu"].mean(0), "std of mu": Lz["mu"].std(0),
                          "mean posterior sigma": Lz["std"].mean(0)}, index=[f"z{i}" for i in range(d)]).round(3)
    stats.to_csv(TABLES / "latent_stats.csv")
    print("prior N(0, I) would give: mean 0, std 1; posterior sigma near 1 means the KL term is active\n",
          stats.to_string())

# ---- every model: its latent codes against the N(0, I) prior ----
from scipy.stats import chi2
rows = []
for j in jobs:
    if not (RUNS / j / "latent.npz").exists():
        continue
    Lj = np.load(RUNS / j / "latent.npz")
    mu = Lj["mu"].astype(float)
    m, cov, corr = mu.mean(0), np.cov(mu, rowvar=False), np.corrcoef(mu, rowvar=False)
    z = np.random.default_rng(0).standard_normal((100_000, mu.shape[1]))          # samples from the prior
    d2 = np.einsum("ij,jk,ik->i", z - m, np.linalg.inv(cov), z - m)              # distance to the encoded data
    rows.append({"job": j, "largest |mean of mu|": np.abs(m).max(), "std of mu (min)": mu.std(0).min(),
                 "std of mu (max)": mu.std(0).max(), "mean posterior sigma": Lj["std"].mean(),
                 "largest |corr| between dims": np.abs(corr - np.eye(len(m))).max(),
                 "prior samples outside the data's 99% region (%)": 100 * np.mean(d2 > chi2.ppf(0.99, len(m)))})
if rows:
    prior = pd.DataFrame(rows).round(3)
    prior.to_csv(TABLES / "latent_vs_prior.csv", index=False)
    print("\nlatent codes against the prior, per model (the region is a Gaussian fitted to the latent means):\n",
          prior.to_string(index=False))

## 5 · Table 3 and Fig. 8: from physical parameters to the latent space

This follows their `latent_space_visualization.ipynb`:
- Keep the curves with all three of absolute G magnitude, T_eff and period, then one row per star (43,031 → 21,796 stars).
- Regress the latent mean μ on the three raw parameters with a linear model (no intercept), a random forest (100 trees) and an MLP (50 tanh units).
- Their dummy variant adds a 4th input ~ U(0,1) and pairs each star with 10 latent samples z ~ N(μ, σ). This keeps some randomness when generating from fixed parameters.

The paper calls its Table 3 numbers validation RMSE on 20% of the data; their notebook scores the fit on the rows it was fitted on. So the table below reports three scores for every regressor, and for both seeds of each model:
- **train RMSE (in-sample)**: scored on the stars the regressor was fitted on;
- **val RMSE (new stars)**: an 80/20 split over stars, so a validation star is never seen in training;
- **val RMSE (split over curves)**: an 80/20 split over all curves, so the augmented copies of one star land on both sides, as in their VAE split.

"Predict the mean" is the RMSE of a regressor that knows nothing; a useful regressor must be clearly below it.

**Fig. 8** feeds RR Lyrae parameters (M_G = −0.24, P = 0.45 d, T_eff from 6000 to 7500 K) through each regressor and then the decoder. With the RF, the paper sees the sawtooth RRab shape turn into the sinusoidal RRc shape as T_eff rises.

In [ ]:
# ---- Table 3 (latent <- physical parameters) and Fig. 8 ----
import warnings
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
warnings.filterwarnings("ignore", category=UserWarning)
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

PHYS2 = ["abs_Gmag", "teff_val", "Period"]
PAPER_T3 = {"tcn": {"Linear": 0.863, "RF": 0.299, "MLP": 0.863}, "tcn_phys": {"Linear": 0.794, "RF": 0.289, "MLP": 0.798}}
REG = {"Linear": lambda: LinearRegression(fit_intercept=False),
       "RF": lambda: RandomForestRegressor(n_estimators=100, max_features=3, n_jobs=-1, random_state=0),
       "MLP": lambda: MLPRegressor(hidden_layer_sizes=(50,), activation="tanh", solver="adam", alpha=1e-4,
                                   learning_rate_init=1e-3, max_iter=200 if not SMOKE else 20, random_state=0)}
rmse = lambda a, b: float(np.sqrt(np.mean((a - b) ** 2)))
tcn_jobs = [j for j in jobs if j.startswith("tcn") and (RUNS / j / "latent.npz").exists()]
t3, fitted = [], {}
for j in tcn_jobs:
    print(f"[{(time.time() - RUN_STARTED) / 3600:5.2f} h] Table 3: fitting the three regressors for {j}", flush=True)
    Lz = np.load(RUNS / j / "latent.npz")
    m = meta.iloc[np.load(RUNS / j / "split.npz")["rows"]].reset_index(drop=True)
    rng = np.random.default_rng(0)
    # (a) one row per star, split over stars: validation stars are never seen in training
    keep = m.dropna(subset=PHYS2).drop_duplicates("OGLE_id").index.values
    P, MU, SD = m.loc[keep, PHYS2].values.astype(float), Lz["mu"][keep].astype(float), Lz["std"][keep].astype(float)
    perm = rng.permutation(len(keep)); ntr = int(0.8 * len(keep))
    itr, iva = perm[:ntr], perm[ntr:]
    # (b) every curve, split over curves: the augmented copies of a star land on both sides (as in their VAE split)
    rows_c = m.dropna(subset=PHYS2).index.values
    Pc, MUc = m.loc[rows_c, PHYS2].values.astype(float), Lz["mu"][rows_c].astype(float)
    perm_c = rng.permutation(len(rows_c)); ntr_c = int(0.8 * len(rows_c))
    ctr, cva = perm_c[:ntr_c], perm_c[ntr_c:]
    ids = m.OGLE_id.values[rows_c]
    shared = np.isin(ids[cva], ids[ctr]).mean()
    # (c) their dummy variant: 10 latent samples per star, a U(0,1) dummy input
    rep = lambda i: (np.hstack([np.repeat(P[i], 10, 0), rng.uniform(size=(10 * len(i), 1))]),
                     rng.normal(np.repeat(MU[i], 10, 0), np.repeat(SD[i], 10, 0)))
    (Ptr_d, Ztr_d), (Pva_d, Zva_d) = rep(itr), rep(iva)
    for name, make in REG.items():
        r = make().fit(P[itr], MU[itr])
        rc = make().fit(Pc[ctr], MUc[ctr])
        rd = make().fit(Ptr_d, Ztr_d)
        fitted[(j, name)] = r
        t3.append({"model": j, "regressor": name, "paper (Table 3)": PAPER_T3["tcn_phys" if "phys" in j else "tcn"][name],
                   "train RMSE (in-sample)": rmse(r.predict(P[itr]), MU[itr]),
                   "val RMSE (new stars)": rmse(r.predict(P[iva]), MU[iva]),
                   "val RMSE (split over curves)": rmse(rc.predict(Pc[cva]), MUc[cva]),
                   "val RMSE, dummy x10": rmse(rd.predict(Pva_d), Zva_d),
                   "predict the mean": rmse(MU[iva], MU[itr].mean(0)), "stars": len(keep),
                   "val curves with a copy in training (%)": round(100 * shared, 1)})
t3 = pd.DataFrame(t3).round(3)
t3.to_csv(TABLES / "table3_regression.csv", index=False)
print("Table 3 — RMSE in latent units (lower is better; 'predict the mean' is the score of knowing nothing)\n",
      t3.to_string(index=False) if len(t3) else "(no latents yet)")

# ---- Fig. 8: RR Lyrae at increasing T_eff (M_G = -0.24, P = 0.45 d), decoded with the cVAE (both seeds) ----
for j in ("tcn", "tcn_s14"):
    if not ((RUNS / j / "model.pt").exists() and (j, "RF") in fitted):
        continue
    vae = tcn_vae.VAE_TCN(4, 600, 5, 48, 9, 3, 0.2, 8, 0).to(DEV)
    vae.load_state_dict(torch.load(RUNS / j / "model.pt", map_location=DEV)); vae.eval()
    teff = np.linspace(6000, 7500, 5)
    pp = np.stack([np.full(5, -0.24), teff, np.full(5, 0.45)], 1)
    dt = torch.linspace(0, 1, 600, device=DEV).repeat(5, 1)
    lab = torch.eye(8, device=DEV)[[classes.index("RRLYR")] * 5]
    fig, ax = plt.subplots(3, 5, figsize=(18, 7), sharex=True, sharey=True)
    for r, name in enumerate(REG):
        z = torch.tensor(fitted[(j, name)].predict(pp), dtype=torch.float32, device=DEV)
        with torch.no_grad():
            lc = vae.decoder(z, dt, lab)[:, :, 0].cpu().numpy()
        for k in range(5):
            ax[r, k].plot(dt[0].cpu().numpy(), lc[k], "k.", ms=2)
            ax[0, k].set_title(f"T_eff = {teff[k]:.0f} K")
        ax[r, 0].set_ylabel(name)
    ax[0, 0].invert_yaxis()
    fig.suptitle(f"{j}: generated RR Lyrae as T_eff rises (their Fig. 8); rows = regressor"); fig.tight_layout()
    fig.savefig(FIGS / f"fig8_rrlyr_teff_{j}.png", dpi=100); plt.show()

## 6 · Can this VAE generate new curves? Sampling from the prior

The paper generates curves through the regression of Table 3. A VAE is also meant to generate by drawing z from the **prior** N(0, I) and decoding it, and that is how the ASTRA benchmark will use a VAE for augmentation. This only works if the encoder's codes fill the prior, which the table above measures. Here we look at it directly, for each seed of the cVAE:
- row 1: z = the latent mean of a real validation curve (a reconstruction, for reference);
- row 2: z drawn from N(0, I), the prior;
- row 3: z drawn from a Gaussian fitted to the latent means of that class (the codes the decoder was trained on).

If rows 2 and 3 look alike, the prior is safe to sample from. If row 2 shows shapes that row 3 never does, the prior has holes.

In [ ]:
# ---- decode latent vectors from three sources: an encoded real curve, the prior, the fitted class Gaussian ----
SHOW = ["CEP", "ECL", "LPV", "RRLYR"]
for j in ("tcn", "tcn_s14"):
    if not (RUNS / j / "model.pt").exists():
        continue
    vae = tcn_vae.VAE_TCN(4, 600, 5, 48, 9, 3, 0.2, 8, 0).to(DEV)
    vae.load_state_dict(torch.load(RUNS / j / "model.pt", map_location=DEV)); vae.eval()
    mu = np.load(RUNS / j / "latent.npz")["mu"]
    sp = np.load(RUNS / j / "split.npz")
    y_all = meta.Type.values[sp["rows"]]
    rng = np.random.default_rng(0)
    dt = torch.linspace(0, 1, 600, device=DEV).repeat(4, 1)
    fig, ax = plt.subplots(3, len(SHOW), figsize=(4 * len(SHOW), 7), sharex=True, sharey=True)
    for k, c in enumerate(SHOW):
        lab = torch.eye(8, device=DEV)[[classes.index(c)] * 4]
        tr_c, va_c = sp["train"][y_all[sp["train"]] == c], sp["val"][y_all[sp["val"]] == c]
        z_rec = mu[rng.choice(va_c, 4, replace=False)]
        z_prior = rng.standard_normal((4, 4))
        z_fit = rng.multivariate_normal(mu[tr_c].mean(0), np.cov(mu[tr_c], rowvar=False), 4)
        for r, z in enumerate((z_rec, z_prior, z_fit)):
            with torch.no_grad():
                lc = vae.decoder(torch.tensor(z, dtype=torch.float32, device=DEV), dt, lab)[:, :, 0].cpu().numpy()
            for i in range(4):
                ax[r, k].plot(dt[0].cpu().numpy(), lc[i], lw=1)
        ax[0, k].set_title(c)
    for r, name in enumerate(("z = encoded real curve", "z ~ N(0, I), the prior", "z ~ Gaussian fitted to the class")):
        ax[r, 0].set_ylabel(name, fontsize=9)
    ax[0, 0].invert_yaxis()
    fig.suptitle(f"{j}: curves decoded from three kinds of latent vectors (4 draws each)"); fig.tight_layout()
    fig.savefig(FIGS / f"prior_samples_{j}.png", dpi=100); plt.show()

In [ ]:
# ---- Deliverables check ----
need = [TABLES / "table1_class_counts.csv", FIGS / "data_examples.png", TABLES / "fig4_losses.csv"]
summary = {"session": SESSION, "smoke": SMOKE, "hours": round((time.time() - RUN_STARTED) / 3600, 2), "jobs": {}}
for j in jobs:
    st = load(RUNS / j / "status.json") or {"state": "not started"}
    summary["jobs"][j] = st
    if st["state"] == "done":
        need += [RUNS / j / f for f in ("model.pt", "history.json", "latent.npz", "recon.npz", "split.npz",
                                         "config.json", "log.txt")]
if all(summary["jobs"].get(j, {}).get("state") == "done" for j in ("tcn", "tcn_phys")):
    need += [TABLES / "table3_regression.csv", TABLES / "latent_vs_prior.csv", FIGS / "fig8_rrlyr_teff_tcn.png",
             FIGS / "fig6_latent.png", FIGS / "prior_samples_tcn.png"]
missing = [str(p.relative_to(WORK)) for p in need if not p.exists()]
json.dump(summary, open(TABLES / f"session{SESSION}_summary.json", "w"), indent=1, default=str)
print(json.dumps(summary["jobs"], indent=1, default=str))
print(f"\noutput size {sum(f.stat().st_size for f in WORK.rglob('*') if f.is_file()) / 1e9:.2f} GB")
print("ALL DELIVERABLES PRESENT" if not missing else f"MISSING: {missing}")
left = [j for j, s in summary["jobs"].items() if s["state"] != "done"]
print("unfinished:", left if left else "none")

## 7 · Observations

Written from session 1 (real run, 4 October 2026, T4 ×2, all five models finished in 6.2 h) and session 2 (11 October 2026, 5 minutes: the results cells only, on the saved models). The paper's Fig. 4 values are read off a plot (about ±0.001). The tables, figures and executed notebooks of both sessions were read.

**Replication verdict.**

| Quantity | Paper | Ours, seed 13 | Ours, seed 14 |
|---|---|---|---|
| Validation loss at epoch 100, cVAE (TCN) | ≈ 0.011 | 0.0228 | **0.0114** |
| Validation loss at epoch 100, cVAE-P (TCN + physics) | ≈ 0.010 | 0.0181 | 0.0140 |
| Validation loss at epoch 100, GRU | ≈ 0.014 | 0.0184 | not run |
| Table 3, cVAE: linear / RF / MLP | 0.863 / 0.299 / 0.863 | 0.880 / 0.893 / 0.884 (new stars) | 0.800 / 0.832 / 0.802 (new stars) |
| Table 3, cVAE-P: linear / RF / MLP | 0.794 / 0.289 / 0.798 | 0.898 / 0.924 / 0.963 (new stars) | 0.855 / 0.866 / 0.853 (new stars) |
| Table 3, predicting the mean latent vector (cVAE / cVAE-P) | not given | 0.883 / 0.901 | 0.802 / 0.852 |

- The cVAE's validation loss is reproduced by one seed of two. Seed 14 gives 0.0114 at epoch 100 and 0.0106 at epoch 150. Seed 13 has its lowest value, 0.0119, at epoch 28 and then gets worse: stopped there it would also be close to the paper.
- The cVAE-P and the GRU end 30–80% above the paper's values in both seeds we have.
- The paper's order "physics < plain TCN < GRU" is not reproduced: our cVAE-P is above our cVAE at epoch 100 in seed 14. The two are trained on different curves (43,031 against 65,760), so their losses are not strictly comparable, in the paper as well as here.
- Table 3: for the cVAE, the linear model and the MLP match the paper on new stars (within 2.5%); for the cVAE-P they are 13% and 21% higher. The random forest matches the paper only **in-sample** (0.330 for the cVAE, 0.344 for the cVAE-P, against 0.299 and 0.289). On stars it has not seen it scores 0.893 and 0.924. In-sample it is 0.315 and 0.322 for the two models of seed 14, so this repeats in all four.
- No regressor beats knowing nothing. On new stars the best regressor and "predict the mean latent vector" score 0.880 and 0.883 (cVAE, seed 13), 0.800 and 0.802 (cVAE, seed 14), 0.898 and 0.901 (cVAE-P, seed 13), 0.853 and 0.852 (cVAE-P, seed 14). The three physical parameters carry almost no information about the latent vector of a star the regressor has not seen.
- The other explanation of the paper's 0.299, a leak through the augmented copies, does not hold. With the split over curves, 62.1% of the validation curves have a copy of their star in training, and the random forest still scores 0.83, 0.90, 0.83 and 0.79 in the four models. Only scoring on the training rows gives a number near the paper's.
- Fig. 8: with the random forest we get a clear sawtooth (RRab) at every T_eff from 6000 to 7500 K, slightly lower in amplitude at the hot end. The change to a sinusoidal RRc shape that the paper shows does not appear. The linear and MLP rows barely react to T_eff. Seed 14 (`fig8_rrlyr_teff_tcn_s14.png`) agrees: its random-forest row is a sawtooth at every T_eff and only jumps in phase between 6750 and 7125 K, and its linear and MLP rows give the same symmetric curve at all five temperatures. Neither seed shows the paper's change of shape.

**Paper vs code.** Following the released code has a visible cost. Its cosine schedule (T_max = 50, stepped with the epoch) brings the learning rate back up to 10⁻³ between epochs 50 and 100. In seed 13 the validation loss starts to rise exactly there (0.0129 at epoch 50, 0.0142 at 70, 0.0228 at 100, 0.0331 at 150) while the training loss keeps falling (0.0141 → 0.0118). The paper's text and Fig. 4 stop at epoch 100; the code trains to 150 and keeps the last weights, with no early stopping.

**Over/under-fitting.** No model under-fits: every training loss falls for the whole run.
- cVAE, seed 13: strong over-fitting after epoch 60 (validation 2.8 × training at the end). Its reconstructions of validation curves have the right shape but are shifted in phase by up to about 0.1 cycle.
- cVAE, seed 14: none. Validation (0.0106) stays below training (0.0128); training is measured with dropout on, validation with dropout off. Its reconstructions follow the validation curves in phase and in shape (`recon_tcn_s14.png`), where seed 13's are shifted.
- cVAE-P: mild in both seeds (lowest validation loss 0.0139 and 0.0135 at epochs 22 and 27, then 0.0178 and 0.0154 at the end).
- Four of the five models have their lowest validation loss between epochs 19 and 28, before the learning rate comes back up; only the cVAE of seed 14 is best at the last epoch.
- GRU: validation between 0.016 and 0.019 and noisy, training still falling.
- Seeds 13 and 14 share the split and differ only in the initial weights and the batch order. So the outcome of the released recipe depends on the seed, and a single run (as in the paper) cannot show that. Why seed 13 over-fits is not established: the phase shifts suggest the encoder memorises where each training curve starts, which cannot carry over to the augmented copies, because those are shifted in phase at random (our reading, not tested).

**Leakage in their split.** 47.5% of the validation stars also have a copy in the training set. The validation loss is therefore not a test on new stars, and the copies are no protection either: seed 13 over-fits although half of its validation stars are "known". A split over stars is needed before this loss can be called generalisation; the ASTRA benchmark splits over stars.

**Latent space.** The latent codes are far from the N(0, I) prior. For the cVAE (seed 13) the means of μ are −0.40, −0.23, −0.12, 0.08 and its spreads 0.38, 0.98, 0.91, 0.78; the first two dimensions are strongly correlated (Fig. 6); the encoder's own σ is 0.02–0.05. With the KL term divided by 10⁴ the model is in practice an autoencoder: each curve gets an almost exact code, and nothing pulls the codes towards the prior. Drawing z from N(0, I) will often land where the decoder was never trained. Session 2 measures it: of 100,000 draws from the prior, 78% fall outside the region that holds 99% of the codes for the cVAE of seed 13, and practically all (99.998%) for seed 14, where one latent dimension has collapsed (the spread of its μ is 0.001). For the cVAE-P it is 78% and 65%, for the GRU 40%. The curves decoded from prior draws (`prior_samples_tcn.png`, `prior_samples_tcn_s14.png`) are smooth and periodic, so the decoder does not break outside the region of the codes. What is lost is variety and class shape. In seed 14 the four prior draws of a class are one template shifted in phase: the same sawtooth for the Cepheids and for the RR Lyrae, and for the eclipsing binaries a sinusoid-like curve without the narrow eclipses that the encoded real curves have. In seed 13 the RR Lyrae draws lose their steep rise and look like the LPV draws. A Gaussian fitted to the class's own codes gives somewhat more varied shapes. So most of what the latent vector carries is where the curve starts in phase. That is a product of the augmentation (random phase shifts), not a property of the star, and it fits Table 3: physical parameters cannot predict it. Consequence for the benchmark: its VAE keeps the full KL term (β = 1) and is judged on curves sampled from the prior, not on reconstructions.

**TCN vs GRU; with vs without physics.** The TCN (seed 14) is clearly better than the GRU (0.0114 against 0.0184 at epoch 100), as in the paper, but the GRU has 46 k parameters against 301 k and trains 6 × faster (0.18 against 1.11 minutes per epoch). The cVAE-P reconstructions follow the data closely, with two visible failures: deep, narrow eclipses come out far too shallow, and gaps in phase are bridged by straight segments. The first is the over-smoothing of sharp features that ASTRA expects from a VAE.

**What this means for ASTRA.**
1. PELS-VAE as released is a good compressor, not yet a generator: it is never sampled from its prior in the paper, and its codes do not fill the prior.
2. Its numbers need more than one seed and a split over stars; ours has two seeds and their split.
3. Table 3's random-forest result is a memorisation effect on our data, so "physical parameters predict the latent vector" is not supported by this replication.